# 1. Introduction and research question

We study whether oil price increases have different effects on financial markets depending on whether they are more closely associated with demand conditions or with a residual non-demand-related component.

The notebook keeps the empirical strategy deliberately simple. The main analysis is monthly, the oil decomposition is reduced-form, and the predictive regressions are interpreted as forecasting relationships rather than full causal identification.

# 2. Imports and setup

We use a short list of standard libraries and a small set of helper functions stored in `src/project_main.py`.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.api import VAR

SRC_PATH = Path("../src")
if not SRC_PATH.exists():
    SRC_PATH = Path("src")
if str(SRC_PATH.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_PATH.resolve()))

from project_main import (
    DAILY_COLUMNS,
    MONTHLY_COLUMNS,
    add_regime_variables,
    aggregate_daily_to_monthly,
    build_project_variables,
    decompose_oil_returns,
    fit_predictive_regression,
    granger_pvalue_table,
    interpret_two_component_model,
    load_data_sheet,
    regression_results_table,
    rolling_forecast_comparison,
    run_adf_table,
)

warnings.filterwarnings("ignore")
plt.style.use("default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_PATH = Path("../data/raw/data_hec_projet_1.xlsx")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/data_hec_projet_1.xlsx")

print("Excel file found:", DATA_PATH.exists())

# 3. Data loading

We load the daily market sheet and the monthly macro sheet, then check the sample ranges.

In [ ]:
daily_raw = load_data_sheet(DATA_PATH, "Daily", DAILY_COLUMNS)  # donnees de marche journalieres
monthly_raw = load_data_sheet(DATA_PATH, "Monthly", MONTHLY_COLUMNS)  # donnees macro mensuelles

print("Daily shape:", daily_raw.shape)
print("Monthly shape:", monthly_raw.shape)
print("Daily range:", daily_raw["date"].min(), "to", daily_raw["date"].max())
print("Monthly range:", monthly_raw["date"].min(), "to", monthly_raw["date"].max())

# 4. Data cleaning and monthly aggregation

We convert the daily market data to one observation per month by keeping the last available observation of each month. We also compute monthly realized S&P 500 volatility from daily log returns.

In [ ]:
monthly_market_data = aggregate_daily_to_monthly(daily_raw)  # passage du daily au monthly
monthly_merged = (
    monthly_market_data.merge(monthly_raw, on="date", how="inner")  # on garde les mois communs
    .sort_values("date")
    .reset_index(drop=True)
)

print("Monthly market shape:", monthly_market_data.shape)
print("Merged monthly shape:", monthly_merged.shape)
display(monthly_merged.head())

# 5. Variable construction

We create monthly log returns for the main asset prices, the term spread, the monthly change in high-yield yield-to-worst, and simple macro growth measures.

In [ ]:
project_df = build_project_variables(monthly_merged)

key_columns = [
    "date",
    "wti_return",
    "brent_return",
    "sp500_return",
    "msci_em_return",
    "hy_change",
    "term_spread",
    "gold_return",
    "sp500_realized_vol",
    "cfnai",
    "ism_mfg",
]

print("Project dataset shape:", project_df.shape)
display(project_df[key_columns].head(12))

# 6. Descriptive statistics and stationarity

We keep the descriptive section short: one summary table, one correlation matrix, one compact figure, and ADF tests.

In [ ]:
descriptive_columns = [
    "wti_return",
    "sp500_return",
    "hy_change",
    "term_spread",
    "gold_return",
    "sp500_realized_vol",
    "cfnai",
    "ism_mfg",
]

summary_table = project_df[descriptive_columns].describe().T
summary_table["skew"] = project_df[descriptive_columns].skew()
summary_table["kurtosis"] = project_df[descriptive_columns].kurtosis()
summary_table = summary_table[["count", "mean", "std", "min", "25%", "50%", "75%", "max", "skew", "kurtosis"]].round(4)

print("Table 1. Summary statistics")
display(summary_table)

print("Table 2. Correlation matrix")
display(project_df[descriptive_columns].corr().round(3))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

series_to_plot = [
    ("wti_return", "Figure 1. Monthly WTI return"),
    ("sp500_return", "Figure 2. Monthly S&P 500 return"),
    ("hy_change", "Figure 3. Monthly HY change"),
]

for ax, (column, title) in zip(axes, series_to_plot):
    ax.plot(project_df["date"], project_df[column])
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
stationarity_columns = ["wti_return", "sp500_return", "hy_change", "term_spread", "gold_return", "cfnai", "ism_mfg"]
adf_table = run_adf_table(project_df, stationarity_columns)

print("Table 3. ADF tests")
display(adf_table)
print("Phillips-Perron is omitted to keep the notebook simple.")

# 7. Oil shock decomposition

We use a simple reduced-form decomposition of monthly WTI returns. The fitted value from a regression on CFNAI is interpreted as the demand-related component, while the residual is interpreted as the non-demand-related component. We also create a simple regime specification based on ISM above or below 50.

In [ ]:
decomposition_df, cfnai_decomp_model = decompose_oil_returns(
    project_df,
    oil_return_col="wti_return",
    activity_col="cfnai",
    prefix="baseline",
)

decomposition_df = add_regime_variables(
    decomposition_df,
    oil_col="wti_return",
    ism_col="ism_mfg",
)

print("Table 4. Oil decomposition with CFNAI")
display(regression_results_table(cfnai_decomp_model))

reconstruction_gap = (
    decomposition_df["wti_return"]
    - decomposition_df["baseline_oil_demand_component"]
    - decomposition_df["baseline_oil_supply_component"]
).dropna()
print("Maximum reconstruction gap:", reconstruction_gap.abs().max())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(decomposition_df["date"], decomposition_df["wti_return"], label="WTI return")
axes[0].plot(decomposition_df["date"], decomposition_df["baseline_oil_demand_component"], label="Demand-related component")
axes[0].set_title("Figure 4. WTI return and fitted demand-related component")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(decomposition_df["date"], decomposition_df["baseline_oil_supply_component"], label="Residual component")
axes[1].set_title("Figure 5. Residual oil component")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 8. Predictive regressions

We estimate one-month-ahead regressions for S&P 500 returns and HY changes. The baseline model uses the demand-related and residual oil components. We also estimate a simple regime-based alternative.

In [ ]:
def print_pair_interpretation(model, variables, labels):
    for variable, label in zip(variables, labels):
        coef = model.params.get(variable, np.nan)
        p_value = model.pvalues.get(variable, np.nan)
        sign = "positive" if coef > 0 else "negative"
        significance = "significant" if p_value < 0.05 else "not significant"
        print(f"{label}: {sign}, {significance}, p-value = {p_value:.4f}")

target_map = {
    "sp500_return": "S&P 500 next-month return",
    "hy_change": "HY next-month change",
}

decomposition_models = {}

for target, label in target_map.items():
    decomp_model, decomp_sample = fit_predictive_regression(
        decomposition_df,
        dependent_col=target,
        predictor_cols=[
            "baseline_oil_demand_component",
            "baseline_oil_supply_component",
            target,
            "term_spread",
            "gold_return",
            "sp500_realized_vol",
        ],
        horizon=1,
        cov_type="HC1",
    )
    decomposition_models[target] = decomp_model

    print(f"Decomposition model: {label}")
    print("Sample size:", len(decomp_sample))
    display(regression_results_table(decomp_model))
    print_pair_interpretation(
        decomp_model,
        ["baseline_oil_demand_component", "baseline_oil_supply_component"],
        ["Demand-related oil effect", "Residual oil effect"],
    )
    print()

In [ ]:
for target, label in target_map.items():
    regime_model, regime_sample = fit_predictive_regression(
        decomposition_df,
        dependent_col=target,
        predictor_cols=[
            "oil_expansion",
            "oil_contraction",
            target,
            "term_spread",
            "gold_return",
            "sp500_realized_vol",
        ],
        horizon=1,
        cov_type="HC1",
    )

    print(f"Regime model: {label}")
    print("Sample size:", len(regime_sample))
    display(regression_results_table(regime_model))

# 9. Granger causality tests

We keep this section limited to the four most relevant pairs in the project.

In [ ]:
granger_specs = [
    ("baseline_oil_demand_component", "sp500_return"),
    ("baseline_oil_supply_component", "sp500_return"),
    ("baseline_oil_demand_component", "hy_change"),
    ("baseline_oil_supply_component", "hy_change"),
]

granger_tables = [granger_pvalue_table(decomposition_df, cause, effect, max_lag=3) for cause, effect in granger_specs]
print("Table 5. Granger causality p-values")
display(pd.concat(granger_tables, ignore_index=True))

# 10. VAR and impulse responses

As a compact extension, we estimate a reduced-form VAR and keep only the two most useful impulse responses.

In [ ]:
var_columns = ["wti_return", "sp500_return", "hy_change", "term_spread", "gold_return", "ism_mfg"]
var_df = decomposition_df[["date"] + var_columns].dropna().set_index("date")

var_model = VAR(var_df)
lag_selection = var_model.select_order(maxlags=6)
lag_choice = lag_selection.selected_orders["bic"]
if lag_choice == 0:
    lag_choice = max(1, lag_selection.selected_orders["aic"])

print("Table 6. VAR lag choice")
display(pd.DataFrame([lag_selection.selected_orders], index=["selected_lag"]))
print("Chosen lag:", lag_choice)

var_results = var_model.fit(lag_choice)

In [ ]:
irf = var_results.irf(12)

fig1 = irf.plot(impulse="wti_return", response="sp500_return")
fig1.set_size_inches(8, 4)
fig1.axes[0].set_title("Figure 6. IRF of S&P 500 return to an oil shock")
plt.tight_layout()
plt.show()

fig2 = irf.plot(impulse="wti_return", response="hy_change")
fig2.set_size_inches(8, 4)
fig2.axes[0].set_title("Figure 7. IRF of HY change to an oil shock")
plt.tight_layout()
plt.show()

# 11. Out-of-sample exercise

We compare a historical-mean benchmark, a raw-oil model, and a decomposition model. We keep only the forecast accuracy table.

In [ ]:
for target, label in target_map.items():
    _, metrics_df = rolling_forecast_comparison(
        decomposition_df,
        target_col=target,
        raw_oil_col="wti_return",
        demand_col="baseline_oil_demand_component",
        supply_col="baseline_oil_supply_component",
        controls=["term_spread", "gold_return", "sp500_realized_vol"],
        start_share=0.6,
    )
    print(f"Forecast comparison: {label}")
    display(metrics_df)

# 12. Robustness checks

We keep the robustness section short and focused: Brent instead of WTI, ISM instead of CFNAI, and MSCI EM instead of the S&P 500.

In [ ]:
robustness_rows = []

brent_df, _ = decompose_oil_returns(decomposition_df, oil_return_col="brent_return", activity_col="cfnai", prefix="brent")
brent_model, brent_sample = fit_predictive_regression(
    brent_df,
    dependent_col="sp500_return",
    predictor_cols=["brent_oil_demand_component", "brent_oil_supply_component", "sp500_return", "term_spread", "gold_return", "sp500_realized_vol"],
    horizon=1,
    cov_type="HC1",
)
robustness_rows.append({
    "check": "Brent instead of WTI",
    "sample_size": len(brent_sample),
    "component_1": brent_model.params.get("brent_oil_demand_component", np.nan),
    "component_1_pvalue": brent_model.pvalues.get("brent_oil_demand_component", np.nan),
    "component_2": brent_model.params.get("brent_oil_supply_component", np.nan),
    "component_2_pvalue": brent_model.pvalues.get("brent_oil_supply_component", np.nan),
})

ism_df, _ = decompose_oil_returns(decomposition_df, oil_return_col="wti_return", activity_col="ism_mfg", prefix="ism")
ism_model, ism_sample = fit_predictive_regression(
    ism_df,
    dependent_col="sp500_return",
    predictor_cols=["ism_oil_demand_component", "ism_oil_supply_component", "sp500_return", "term_spread", "gold_return", "sp500_realized_vol"],
    horizon=1,
    cov_type="HC1",
)
robustness_rows.append({
    "check": "ISM instead of CFNAI",
    "sample_size": len(ism_sample),
    "component_1": ism_model.params.get("ism_oil_demand_component", np.nan),
    "component_1_pvalue": ism_model.pvalues.get("ism_oil_demand_component", np.nan),
    "component_2": ism_model.params.get("ism_oil_supply_component", np.nan),
    "component_2_pvalue": ism_model.pvalues.get("ism_oil_supply_component", np.nan),
})

em_model, em_sample = fit_predictive_regression(
    decomposition_df,
    dependent_col="msci_em_return",
    predictor_cols=["baseline_oil_demand_component", "baseline_oil_supply_component", "msci_em_return", "term_spread", "gold_return", "sp500_realized_vol"],
    horizon=1,
    cov_type="HC1",
)
robustness_rows.append({
    "check": "MSCI EM instead of S&P 500",
    "sample_size": len(em_sample),
    "component_1": em_model.params.get("baseline_oil_demand_component", np.nan),
    "component_1_pvalue": em_model.pvalues.get("baseline_oil_demand_component", np.nan),
    "component_2": em_model.params.get("baseline_oil_supply_component", np.nan),
    "component_2_pvalue": em_model.pvalues.get("baseline_oil_supply_component", np.nan),
})

print("Table 7. Robustness checks")
display(pd.DataFrame(robustness_rows).round(4))

# 13. Main conclusions

The main message of the notebook should be read in a reduced-form sense. The fitted oil component is interpreted as demand-related, while the residual component is interpreted as non-demand-related. The project is designed to provide a transparent and coherent empirical framework rather than a full structural identification strategy.

In [ ]:
print("Main takeaway for S&P 500 returns")
for line in interpret_two_component_model(
    decomposition_models["sp500_return"],
    "baseline_oil_demand_component",
    "baseline_oil_supply_component",
):
    print(line)

print("\nMain takeaway for HY changes")
for line in interpret_two_component_model(
    decomposition_models["hy_change"],
    "baseline_oil_demand_component",
    "baseline_oil_supply_component",
):
    print(line)